In [17]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
import matplotlib.pyplot as plt
%matplotlib widget


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)
    ### RETURNS: x = log_{10}(L_bol/L_solar)  (solar L_bol==3.9x10^33 erg/s adopted here)
    ###          y = log_{10}(dphi(L_bol)/dlog(L_bol))   [Mpc^-3 log(L_bol)^{-1}]
    ###          yerr = +/- 1sigma uncertainty in log_{10}(dphi(L_bol)/dlog(L_bol))



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams_ln()

        
        self.max_halo = np.log(1.0e15)
        self.min_halo = np.log(1.0e7)
        self.HaloBins = np.linspace(self.min_halo, self.max_halo, bin_num)
        
        slopes = self.get_slope_ln(self.HaloBins)
        
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(self.min_halo, self.max_halo, bin_num)
            slopes = self.get_slope_ln(self.HaloBins)

            
        self.max_stell = self.get_Mstar_ln(self.max_halo)
    
        self.fp = self.HaloBins
        self.xp = self.get_Mstar_ln(self.fp)
        
        self.LumBins = np.linspace(np.log(1.0e5), np.log(1.0e16), bin_num)
        self.StellBins = np.linspace(np.log(1.0e5),self.max_stell, bin_num)
        
    
        
    def get_zparams_ln(self):
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = np.e**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
        
    
    def get_slope_ln(self, Mhalo):

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*np.e**(self.zparams['beta']*dm)+self.zparams['beta']*np.e**(self.zparams['alpha']*dm))/(np.e**(self.zparams['beta']*dm) + np.e**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    
    def get_Mstar_ln(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log(np.e**(-self.zparams['alpha']*dm) + np.e**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.e**(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [np.log(1.0e7), np.log(1.4e4)]
        stop = [np.log(1.0e12), np.log(1.4e9)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope_ln(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < self.min_halo] = self.min_halo
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(np.e**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')
        meanMstar = np.apply_along_axis(self.get_Mstar_ln, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(np.e**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log(3.3e4) - Mbh

        return n
        
    
    def get_mean_etas(self, vals, a, xsigs, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        if slope == self.slope_list[0]:
            xsig = xsigs[0]
        else:
            xsig = xsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - np.log10(np.e**Mstar)))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3*10**31 * np.e**Mbh #J/s
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(np.e**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        
        
        Mdotbhmean = -0.5*xsig**2 + np.log(Mdotbh/(2*10**30))
        etamean = Mdotbhmean - np.log(Mdotedd/(2*10**30))
        nsig = xsig
        
        return etamean, nsig, Mdotbh, Mdotedd
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2], vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))
        
        self.eta_lists = eta_lists
        
        vals = np.zeros((self.bin_num,3))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a, xsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        
        
        self.plotpurp = mean_etas
        
        self.dNdL = np.log((1-obscured) * (np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
        ind_dNdL_off = (1-obscured) * np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdL_off:
            self.ind_dNdL[c,:] = l
            c += 1

bins = 500
fig = plt.figure(figsize=(5,5))
z=3
obscured = .8
sig_X = [1.6,.65]
dM = 0.55
mMid = 10.3
smhm_scat = 0.6
qlf = QLF(z, bins)
qlf.get_dNdMstar(smhm_scat)
qlf.get_SMBM(dM, mMid)
qlf.get_dNdMbh()
qlf.get_dNdL(sig_X, obscured)
col = np.zeros(qlf.bin_num)
col[qlf.early] = 0
col[qlf.growth] = 1
col[qlf.late] = 2


plt.plot(qlf.LumBins, qlf.dNdL,c='k')
plt.axis([5.8,16,-10,0])
plt.axis([10,38,-30,0])
x, y , yerr = grab_obs(z)
plt.errorbar(x, y, yerr = yerr, fmt = 'o', markersize = .5,color='blue')
for l, c in zip(qlf.ind_dNdL, col):
    c_list = ['r','gray','gray']
    plt.plot(qlf.LumBins, np.log(l), linewidth = .5, color = c_list[int(c)])


FigureCanvasNbAgg()

$    L_{Edd} = const \frac{M_{BH}}{M_{\odot}} = \epsilon \dot{M}_{Edd} c^2    $

$    \frac{dM_{BH}}{dt} = \frac{M_{BH}}{M_{*}} \frac{dlogM_{BH}}{dlogM_{*}} \frac{dM_{*}}{dt}    $

$    \eta = \frac{\dot{M}_{BH}}{\dot{M}_{Edd}}    $

$    X = \frac{\dot{M}_{BH}}{<\dot{M}_{BH}>}    $

$    <\dot{M}_{BH}> = M_{BH} \frac{dlogM_{BH}}{dlogM_{*}} \frac{<\dot{M}_*>}{M_*}    $

$    <\eta> = \frac{<\dot{M}_{BH}>}{\dot{M}_{Edd}}      $



$ \mu_{logX} = \mu_{lnX} log(e),\ \ \ \ \ \ \sigma_{logX} = \sigma_{lnX} log(e)    $

$ log\dot{M}_{BH} = logX + log<\dot{M}_{BH}>,\ \ \ \ \ \ \mu_{log\dot{M}_{BH}} = \mu_{logX} + log<\dot{M}_{BH}>,\ \ \ \ \ \ \sigma_{log\dot{M}_{BH}} = \sigma_{logX}   $

$ log\eta = log\dot{M}_{BH} - log\dot{M}_{Edd},\ \ \ \ \ \ \mu_{log\eta} = \mu_{log\dot{M}_{BH}} - log\dot{M}_{Edd},\ \ \ \ \ \ \sigma_{log\eta} = \sigma_{log\dot{M}_{BH}} = \sigma_{logX}  $

In [240]:
def gauss(x, var):

    mean, std, amp = var
    y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

    return y

params = [0, np.sqrt(.2), 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)

params = [0, 1, 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)

params = [0, np.sqrt(5.), 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)


xn = x/2
plt.plot(xn,y)
print(np.var(x), np.var(xn))


8.41708542713568 2.10427135678392


Code to create universal $<\dot{M}_{BH}>$, z evolution plot.

In [247]:
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib
plt.close('all')
bins = 200
fig, ax = plt.subplots(figsize=(6,6))
cbax = fig.add_axes([0.05, 0.35, .9, 0.05]) 
fig.subplots_adjust(bottom=0.5)
dM = .5
obscured = .75
zlist = np.linspace(0,6,500)
mulist = []
cut = []
for z in zlist:
    obscured = .75
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(0.3)
    qlf.get_SMBM(dM, 10.3)
    qlf.get_dNdMbh()
    qlf.get_dNdL([.8,.4], obscured)
    mu = qlf.plotpurp[:,0]
    mulist.append(mu)
    cut.append(np.argmin(qlf.early))

cutlist = []
for i, mul in zip(cut,mulist):
    cutlist.append(mul[i])
    
norm = matplotlib.colors.Normalize(vmin=min(qlf.StellBins), vmax=max(qlf.StellBins))
cb = matplotlib.colorbar.ColorbarBase(cbax, cmap='inferno', norm=norm, orientation='horizontal')
cb.set_label(r'$log_{10}[M_{*}]$')

i_mulist = np.transpose(mulist)
for i, j in zip(i_mulist,qlf.StellBins):
    ax.plot(zlist, i, c = cm.inferno(norm(j)),linewidth = 0.5)

ax.set_ylabel(r'$log_{10} [<\dot{M}_{BH}>/M_{\odot}] yr^{-1}$')
ax.set_xlabel(r'z')
ax.plot(zlist,cutlist,c='k',label='critical mass',linestyle='dotted')
ax.legend()
fig.suptitle(r'Redshift evolution of <$\dot{M}_{BH}$> for varrying $M_{*}$')

fig.show()
#fig.savefig('plots/Mdot_z_evolution.pdf')


FigureCanvasNbAgg()

/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:231: RuntimeWarning: divide by zero encountered in log10
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:233: RuntimeWarning: divide by zero encountered in log10


In [257]:
#zlist = [0.0, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
zlist = [0.0,1.0,2.0,3.0,4.0,5.0,6.0]
x = np.linspace(-10,3,500)
obscured = .75
dM = .5
xsig1 = .8
xsig2 = .4
plt.close('all')
count = 1
bigsumg1 = np.zeros(500)
bigsumg2 = np.zeros(500)
bins = 200
for z, c in zip(zlist,['red','gold','mediumspringgreen','deepskyblue','navy','darkorchid','palevioletred','lightcoral','olive','k','gray','aqua','pink','saddlebrown','lime']):
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(0.3)
    qlf.get_SMBM(dM, 10.3)
    qlf.get_dNdMbh()
    qlf.get_dNdL([.8,.4], obscured)
    mu = qlf.plotpurp[:,0]
    xsig = qlf.plotpurp[:,1]
    s1count = 0
    s2count = 0
    sumg1 = np.zeros(500)
    sumg2 = np.zeros(500)
    for m, s in zip(mu,xsig):
        if s == xsig1:
            sumg1 += qlf.gauss(x, m, s, 1)
            s1count += 1
        elif s == xsig2:
            sumg2 += qlf.gauss(x, m, s, 1)
            s2count += 1
    bigsumg1 += sumg1/s1count
    bigsumg2 += sumg2/s2count
#     if count == 1:
#         plt.plot(x,sumg1/s1count,linestyle = 'dashed',color='k', label = 'pre crit mass')
#         plt.plot(x,sumg2/s2count,linestyle = 'solid', label = 'post crit mass',color='k')
#         count+=1
    plt.plot(x,sumg1/s1count,linestyle = 'dashed',color=c,linewidth=1)
    plt.plot(x,sumg2/s2count,linestyle = 'solid', label = 'z = '+str(z),color=c,linewidth=1)
plt.plot(x,bigsumg1/3,linestyle = 'dashed', color = 'k', label = 'pre crit mass')
plt.plot(x,bigsumg2/3,linestyle='solid',c='k', label = 'post crit mass')
    
plt.legend(fontsize=8)
plt.xlabel(r'$log_{10} [<\dot{M}_{BH}>/M_{\odot}] yr^{-1}$')
plt.ylabel(r'probability density')
plt.title(r'$<\dot{M}_{BH}>$ distributions')
plt.savefig('plots/Mdot_distributions.pdf')


FigureCanvasNbAgg()

Code to create universal $<\dot{M}_{BH}>$ model.

In [16]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.HaloBins = np.linspace(7., self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(7., self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)
        
        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self): ##converting this to ln????
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
        
        
    
    def get_slope(self, Mhalo): ##converting this to ln????

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo): ##converting this to ln????
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < 7.] = 7.
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(10.**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16') * np.log(10)
        meanMstar = np.apply_along_axis(self.get_Mstar, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')  / (get_slope(um.get_Mhalo(self.StellBins, self.z)) * np.log10(np.e))
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log10(3.3e4) - Mbh

        return n
        
    
    def get_mean_etas(self, vals, a, xsigs, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        if slope == self.slope_list[0]:
            xsig = xsigs[0]
        else:
            xsig = xsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3*10**31 * 10**Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(10**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        
        
        Mdotbhmean = -0.5*xsig**2*np.log10(np.e)+np.log10(Mdotbh/(2*10**30))
        etamean = Mdotbhmean - np.log10(Mdotedd/(2*10**30))
        
        nsig = xsig * np.log10(eta)
        
        return etamean, nsig, Mdotbh, Mdotedd
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2], vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        leftc = np.argmin(self.early)
        rightc = np.argmax(self.late)
        per = len(self.growth[self.growth==True])*.1
        cut1l = int(leftc - per)
        cut1r = int(leftc + per + 1)
        cut2l = int(rightc - per)
        cut2r = int(rightc + per + 1)

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))
        
        self.eta_lists = eta_lists
        
        vals = np.zeros((self.bin_num,3))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a, xsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        
        
        self.plotpurp = mean_etas
        
        self.dNdL = np.log10((1-obscured) * (np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
        ind_dNdL_off = (1-obscured) * np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdL_off:
            self.ind_dNdL[c,:] = l
            c += 1

bins = 500
fig = plt.figure(figsize=(5,5))
z=3
obscured = .8
sig_X = [.85,.35]
dM = 0.55
mMid = 10.3
smhm_scat = 0.3
qlf = QLF(z, bins)
qlf.get_dNdMstar(smhm_scat)
qlf.get_SMBM(dM, mMid)
qlf.get_dNdMbh()
qlf.get_dNdL(sig_X, obscured)
col = np.zeros(qlf.bin_num)
col[qlf.early] = 0
col[qlf.growth] = 1
col[qlf.late] = 2


plt.plot(qlf.LumBins, qlf.dNdL,c='k')
plt.axis([5.8,16,-10,0])
x, y , yerr = grab_obs(z)
plt.errorbar(x, y, yerr = yerr, fmt = 'o', markersize = .5,color='blue')
for l, c in zip(qlf.ind_dNdL, col):
    c_list = ['r','gray','gray']
    plt.plot(qlf.LumBins, np.log10(l), linewidth = .5, color = c_list[int(c)])


FigureCanvasNbAgg()

/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:301: RuntimeWarning: divide by zero encountered in log10


In [7]:
plt.close('all')
import matplotlib.gridspec as gridspec

def lognorm(x, sig, mean, loc):
    y = np.exp(-(np.log(x-loc)-mean)**2/(2*sig**2))/((x-loc)*sig*np.sqrt(2*np.pi))
    return y

def get_mu(mean, sig):
    return np.log(mean)-0.5*sig**2

def get_mean(mu, sig):
    return np.exp(mu+0.5*sig**2)

def get_variance(mu, sig):
    return (np.exp(sig**2)-1)*np.exp(2*mu + sig**2)


gs = gridspec.GridSpec(2, 2)
fig = plt.figure(figsize=(12,12))
ax = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0,1])
ax2 = fig.add_subplot(gs[1,1])
ax3 = fig.add_subplot(gs[1,0])


s1 = .55
s2 = .85
x1 = np.linspace(-12,12,20000)
x2 = np.linspace(0,10,200000)
x3 = np.linspace(-8,0,200000)



ax3.plot(x1,qlf.gauss(x1, -0.5*s1**2*np.log10(np.e), s1*np.log10(np.e), 1),c='k')
ax3.plot(x1,qlf.gauss(x1, -0.5*s2**2*np.log10(np.e), s2*np.log10(np.e),1),c='r')
ax3.axis([-2,2,0,1.8])
ax3.set_xlabel(r'$log_{10}[X]$')
ax3.set_ylabel('Probability Density')
ax3.text(1,1.,r'$\sigma_{log_{10}X} = $'+str(s1*np.log10(np.e))[0:4],color='k')
ax3.text(1,.9,r'$\sigma_{log_{10}X} = $'+str(s2*np.log10(np.e))[0:4],color='r')
ax3.text(1,1.2,r'$\mu_{log_{10}X} = $'+str(-0.5*s1**2*np.log10(np.e))[0:5], color='k')
ax3.text(1,1.1,r'$\mu_{log_{10}X} = $'+str(-0.5*s2**2*np.log10(np.e))[0:5],color='r')
# ax3.text(-1.9,1.25,r'$\mu_{log_{10}[X]} = 0$')
# ax3.text(-1.9,1.35,r'$\sigma_{log_{10}[X]}^2 = e^{2\mu_X +\sigma_X ^2}(e^{\sigma_X ^2}-1) / ln[10]^2$')
# ax3.text(-1.9,1.5,r'$p(log_{10}[X]) = \frac{1}{\sqrt{2\pi\sigma_{log_{10}[X]}^2}}e^{-(log_{10}[X]-\mu_{log_{10}[X]})^2/(2\sigma_{log_{10}[X]}^2)}$')

print(lognorm(x2, s1, -0.5*s1**2, 0))

ax.plot(x2, lognorm(x2, s1, -0.5*s1**2, 0), c = 'k', label=r'pre-disk')
ax.plot(x2, lognorm(x2, s2, -0.5*s2**2, 0), c= 'r', label=r'post-disk')
ax.axis([0,6,0,1.5])
ax.set_xlabel(r'$X$')
ax.set_ylabel('Probability Density')
ax.text(2,1.,r'$\sigma_{lnX} = $'+str(s1),color='k')
ax.text(2,.9,r'$\sigma_{lnX} = $'+str(s2),color='r')
ax.text(2,1.1,r'$\mu_{lnX} = - \sigma_{lnX}/2$')
# ax.text(2,1.1,r'$\mu_{lnX} = ?$')
# ax.text(-2.8,1,r'$p(X) = \frac{e^{-(ln(X)-\mu_{lnX})^2/(2\sigma_{lnX} ^2)}}{X\sigma_{lnX}\sqrt{2 \pi}}$',fontsize=13)
# ax.text(-2.7,.9,r'$<X> \equiv 1$')
# ax.text(-2.8,.8,r'$<X> = \int_{0}^{\infty}$ X p(X) dX')
# ax.text(-1.8,.7,r'$= e^{\mu_{lnX} + \sigma_{lnX}/2}$')
# ax.text(-2.4,.6,r'$\mu_{lnX} = ln[1] - \sigma_{lnX}/2$')
# ax.text(-1.8,.5, r'$ = - \sigma_{lnX}/2$')
ax.legend(loc='upper center', bbox_to_anchor=(0.75, .98))
ax.axvline(1, color = 'gray', linestyle='dotted')

plt.suptitle(r'$X = \frac{\dot{M}_{BH}}{<\dot{M}_{BH}>}$     Universal Relation')



ax1.plot(x1,qlf.gauss(x1, -0.5*s1**2, s1, 1),c='k')
ax1.plot(x1,qlf.gauss(x1, -0.5*s2**2, s2,1),c='r')
ax1.axis([-2,2,0,1.8])
ax1.set_xlabel(r'$ln[X]$')
ax1.set_ylabel('Probability Density')
ax1.text(1,1.,r'$\sigma_{lnX} = $'+str(s1)[0:4],color='k')
ax1.text(1,.9,r'$\sigma_{lnX} = $'+str(s2)[0:4],color='r')
ax1.text(1,1.2,r'$\mu_{lnX} = $'+str(-0.5*s1**2)[0:5], color='k')
ax1.text(1,1.1,r'$\mu_{lnX} = $'+str(-0.5*s2**2)[0:5],color='r')

for mdot, c in zip([-1,-2,-3,-4,-5],['r','orange','gold','g','b']):
    ax2.plot(x1, qlf.gauss(x1, -0.5*s1**2*np.log10(np.e)+mdot, s1*np.log10(np.e), 1),c=c,linestyle='dashed',label = r'pre-$log_{10}[<\dot{M}_{BH}>] = $'+str(mdot))
    ax2.plot(x1, qlf.gauss(x1, -0.5*s2**2*np.log10(np.e)+mdot, s2*np.log10(np.e), 1),c=c,linestyle='solid',label = r'post-$log_{10}[<\dot{M}_{BH}>] = $'+str(mdot))

ax2.axis([-10,0,0,1.8])
ax2.set_xlabel(r'$log_{10}[\dot{M}_{BH}] \ \ yr^{-1}$')
ax2.set_ylabel('Probability Density')
ax2.legend()

fig.savefig('plots/universaldist_mdot_v2.3.pdf')

FigureCanvasNbAgg()

/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: divide by zero encountered in log
  """
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: divide by zero encountered in true_divide
  """
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: invalid value encountered in true_divide
  """


[           nan 7.77059197e-65 8.89971498e-56 ... 3.45334061e-06
 3.45318328e-06 3.45302595e-06]


In [533]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.HaloBins = np.linspace(7., self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(7., self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)
        
        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self):
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
    
    def get_slope(self, Mhalo):

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < 7.] = 7.
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(10.**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16') * np.log(10)
        meanMstar = np.apply_along_axis(self.get_Mstar, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')  / (get_slope(um.get_Mhalo(self.StellBins, self.z)) * np.log10(np.e))
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log10(3.3e4) - Mbh

        return n
        
        
        
            
#     def get_mean_etas(self, vals, a, xsigs, files = files):

#         Mbh = vals[0]
#         Mstar = vals[1]
#         slope = vals[2]
#         if slope == self.slope_list[0]:
#             xsig = xsigs[0]
#         else:
#             xsig = xsigs[1]
#         closest_a = np.argmin(np.abs(a_list - a))
#         closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
#         ssfr = ssfr_list[closest_a][closest_m]

#         Ledd = 1.3*10**31 * 10**Mbh #J/s 
#         Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
#         sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
#         Mdotbh = sbhr*(10**Mbh*2*10**30)
#         eta = Mdotbh/Mdotedd
#         nsig = xsig * np.log10(eta)
        
#         return np.log10(eta), nsig
    
    def get_mean_etas(self, vals, a, xsigs, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        if slope == self.slope_list[0]:
            xsig = xsigs[0]
        else:
            xsig = xsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        masses = np.array(mass_list[closest_a])
        ssfrs = np.array(ssfr_list[closest_a])
        closest_m = np.argmin(np.abs(masses - Mstar))
        nonzero = (ssfrs != 0)
        minm = np.min(masses[nonzero])
        maxm = np.max(masses[nonzero])
        if minm < Mstar < maxm:
            ssfr = np.interp(Mstar, masses[nonzero], ssfrs[nonzero])
        else:
            ssfr = ssfr_list[closest_a][closest_m]
        
        flag = 0
        if Mstar > maxm:
            flag = 1


        Ledd = 1.3*10**31 * 10**Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(10**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        nsig = xsig * np.log10(eta)
        
        return np.log10(eta), nsig, Mdotbh, Mdotedd, Mstar, ssfr, flag, Mbh, sbhr
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2]-(vals[-1]**2)/2, vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        leftc = np.argmin(self.early)
        rightc = np.argmax(self.late)
        per = len(self.growth[self.growth==True])*.1
        cut1l = int(leftc - per)
        cut1r = int(leftc + per + 1)
        cut2l = int(rightc - per)
        cut2r = int(rightc + per + 1)

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))
        
        self.eta_lists = eta_lists
        
        vals = np.zeros((self.bin_num,3))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a, xsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        
        
        self.plotpurp = mean_etas
        
        self.dNdL = np.log10((np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
        ind_dNdL_off = (1-obscured) * np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdL_off:
            self.ind_dNdL[c,:] = l
            c += 1


In [551]:
plt.close('all')
bins = 50
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(1,5,figsize=(30,5))
obscured = .8
sig_X = [1.25,.85]
dM = .55
mMid = 10.3
smhm_scat = 0.3
for z, c in zip([0.5,1.0,1.5,2.0,2.5,3.0,4.0],['r','orange','gold','green','blue','violet','indigo','']):
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(smhm_scat)
    qlf.get_SMBM(dM, mMid)
    qlf.get_dNdMbh()
    qlf.get_dNdL(sig_X, obscured)

    logMstar = qlf.plotpurp[:,4]
    ssfr = qlf.plotpurp[:,5] #yr^-1
    sfr = ssfr * 10**logMstar
    Mdotbh = qlf.plotpurp[:,2]/(2*10**30*3.17098e-8)
    flags = qlf.plotpurp[:,6]
    logMbh = qlf.plotpurp[:,7]
    sbhr = qlf.plotpurp[:,8]
    lw = 1. 
    
    ax1.plot(logMstar, np.log10(Mdotbh),label='z = '+str(z),color = c,linewidth = lw)
    ax1.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')
    ax1.set_ylabel(r'$log_{10}[<\dot{M}_{BH}>] \ \ yr^{-1}$')
    
    ax2.plot(np.log10(sfr), np.log10(Mdotbh),label='z = '+str(z),color = c,linewidth = lw)
    ax2.set_xlabel(r'$log_{10}[<\dot{M}_{*}>] \ \ yr^{-1}$')
    ax2.set_ylabel(r'$log_{10}[<\dot{M}_{BH}>] \ \ yr^{-1}$')
    
    ax3.plot(logMstar, logMbh, color = c, linewidth = lw)
    ax3.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')
    ax3.set_ylabel(r'$log_{10}[M_{BH}/M_{\odot}]$')
    
    ax4.plot(logMstar, logMbh/logMstar, color = c, linewidth = lw)
    ax4.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')    
    ax4.set_ylabel(r'$log_{10}[M_{BH}/M_{*}]$')
    
    ax5.plot(logMstar, np.log10(sfr), color = c, linewidth = lw)
    ax5.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')
    ax5.set_ylabel(r'$log_{10}[<\dot{M}_{*}>] \ \ yr^{-1}$')
    
plt.tight_layout()
ax1.legend()
fig.savefig('plots/MdotBH_Mstar_SFR_relation.pdf')

FigureCanvasNbAgg()

In [12]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
from scipy.optimize import newton


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.HaloBins = np.linspace(7., self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(7., self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)
        
        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self):
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
    
    def get_slope(self, Mhalo):

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < 7.] = 7.
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(10.**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16') * np.log(10)
        meanMstar = np.apply_along_axis(self.get_Mstar, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')  / (get_slope(um.get_Mhalo(self.StellBins, self.z)) * np.log10(np.e))
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log10(3.3e4) - Mbh

        return n
        
        
        
            
    def get_mean_etas(self, vals, a, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        xsig = vals[3]
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3*10**31 * 10**Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(10**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        nsig = xsig * np.log10(Mdotbh)/np.log10(Mdotedd)
        
        return np.log10(eta), nsig
    
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2]-(vals[-1]**2)/2, vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, dX, obscured):
        
        lcut = np.argmin(np.abs(self.StellBins - self.mass_cuts[0] - dX))
        rcut = np.argmin(np.abs(self.StellBins - self.mass_cuts[0] + dX))
        s = np.zeros(self.bin_num)
        s[:lcut] = xsigs[0]
        s[lcut:rcut+1] = np.linspace(xsigs[0],xsigs[1],len(s[lcut:rcut+1]))
        s[rcut+1:] = xsigs[1]
        
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))

        vals = np.zeros((self.bin_num,4))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        vals[:,3] = s
        
           
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        print(mean_etas[:,1])
        
        self.dNdL = np.log10((1-obscured) * (np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
  

In [13]:
plt.close('all')
import matplotlib.pyplot as plt
bins = 500
fig = plt.figure(figsize=(5,5))
z=2
obscured = .8
sig_X = [1.25,.85]
dM = 0.55
mMid = 10.3
smhm_scat = 0.3
dX = 0.2
qlf = QLF(z, bins)
qlf.get_dNdMstar(smhm_scat)
qlf.get_SMBM(dM, mMid)
qlf.get_dNdMbh()
qlf.get_dNdL(sig_X, dX, obscured)
xm, ym = qlf.LumBins, qlf.dNdL

plt.plot(xm,ym,c='k')
x, y , yerr = grab_obs(z)
plt.errorbar(x, y, yerr = yerr, fmt = 'o', markersize = .5,color='blue')

FigureCanvasNbAgg()

[1.12738415 1.12740158 1.12741901 1.12743644 1.12745386 1.12747128
 1.12748869 1.1275061  1.1275235  1.12754089 1.12755829 1.12757567
 1.12759305 1.12761043 1.1276278  1.12764517 1.12766253 1.12767989
 1.12769724 1.12771459 1.12773193 1.12774927 1.1277666  1.12778393
 1.12780125 1.12781857 1.12783588 1.12785319 1.12787049 1.12788779
 1.12790508 1.12792237 1.12793966 1.12795693 1.12797421 1.12799148
 1.12800874 1.128026   1.12804325 1.1280605  1.12807775 1.12809499
 1.12811222 1.12812945 1.12814668 1.1281639  1.12818111 1.12819832
 1.12821553 1.12823273 1.12824992 1.12826711 1.1282843  1.12830148
 1.12831866 1.12833583 1.12835299 1.12837016 1.12838731 1.12840447
 1.12842161 1.12843875 1.12845589 1.12847303 1.12849015 1.12850728
 1.12852439 1.12854151 1.12855862 1.12857572 1.12859282 1.12860991
 1.128627   1.12864409 1.12866117 1.12867824 1.12869531 1.12871238
 1.12872944 1.12874649 1.12876354 1.12878059 1.12879763 1.12881467
 1.1288317  1.12884873 1.12886575 1.12888277 1.12889978 1.1289

<ErrorbarContainer object of 3 artists>